In [1]:
import os
import databento as db
from dotenv import load_dotenv

load_dotenv('./.env')

client = db.Historical(os.getenv("DATABENTO_API_KEY"))  # uses DATABENTO_API_KEY env var
db_trades = client.timeseries.get_range(
    dataset="OPRA.PILLAR",
    schema="cmbp-1",                
    # symbols=["SPY.OPT"], 
    # stype_in="parent",                   # request the chain; filter in code
    stype_in="raw_symbol",
    symbols=["SPY   230830P00423000"],
    start="2023-08-22T14:29:59.990000Z",
    end="2023-08-22T14:30:00.500000Z",
    limit=100
).to_df()
print(db_trades.shape)
display(db_trades.head())
print(db_trades.symbol.unique())

(8, 17)


,ts_event,rtype,publisher_id,instrument_id,action,side,price,size,flags,ts_in_delta,bid_px_00,ask_px_00,bid_sz_00,ask_sz_00,bid_pb_00,ask_pb_00,symbol
ts_recv,,,,,,,,,,,,,,,,,
2023-08-22 14:29:59.991083538+00:00,2023-08-22 14:29:59.990872832+00:00,177,30,654312856,A,N,0.49,271,194,0,0.48,0.49,1497,271,0,0,SPY 230830P00423000
2023-08-22 14:30:00.048469173+00:00,2023-08-22 14:30:00.048260352+00:00,177,30,654312856,A,N,0.49,271,194,0,0.48,0.49,1670,271,0,0,SPY 230830P00423000
2023-08-22 14:30:00.075664939+00:00,2023-08-22 14:30:00.075455488+00:00,177,30,654312856,A,N,0.49,321,194,0,0.48,0.49,1670,321,0,0,SPY 230830P00423000
2023-08-22 14:30:00.106431944+00:00,2023-08-22 14:30:00.106223616+00:00,177,30,654312856,A,N,0.49,321,194,0,0.48,0.49,1423,321,0,0,SPY 230830P00423000
2023-08-22 14:30:00.449679234+00:00,2023-08-22 14:30:00.449470208+00:00,177,30,654312856,A,N,0.49,271,194,0,0.48,0.49,1423,271,0,0,SPY 230830P00423000


['SPY   230830P00423000']


In [2]:
import pandas as pd

trades_df = pd.read_parquet('../rust-ingest/pcap_samples/ny4-small-10k_mbp.parquet')
print(trades_df.shape)
display(trades_df.head())
display(trades_df.tail())

(14440, 19)


,ts_event,ts_recv,ts_event_utc,rtype,publisher_id,instrument_id,action,side,price,size,flags,ts_in_delta,bid_px_00,ask_px_00,bid_sz_00,ask_sz_00,bid_pb_00,ask_pb_00,symbol
0,1692714599999800576,1692714599999800576,2023-08-22T14:29:59.999800576Z,177,30,NaN,A,A,0.50,418.0,NaN,0,0.48,0.50,310,418,0,0,SPY 230830P00423000
1,1692714599999800576,1692714599999800576,2023-08-22T14:29:59.999800576Z,177,30,NaN,A,A,0.80,434.0,NaN,0,0.78,0.80,45,434,0,0,SPY 230822P00439000
2,1692714599999800576,1692714599999800576,2023-08-22T14:29:59.999800576Z,177,30,NaN,A,A,2.28,193.0,NaN,0,2.25,2.28,121,193,0,0,SPY 230825P00438000
3,1692714599999804416,1692714599999804416,2023-08-22T14:29:59.999804416Z,177,30,NaN,A,A,17.90,212.0,NaN,0,17.76,17.90,207,212,0,0,SPY 240315C00455000
4,1692714599999802624,1692714599999802624,2023-08-22T14:29:59.999802624Z,177,30,NaN,A,A,0.94,52.0,NaN,0,0.92,0.94,350,52,0,0,SPY 230823C00442000


,ts_event,ts_recv,ts_event_utc,rtype,publisher_id,instrument_id,action,side,price,size,flags,ts_in_delta,bid_px_00,ask_px_00,bid_sz_00,ask_sz_00,bid_pb_00,ask_pb_00,symbol
14435,1692714600031779072,1692714600031779072,2023-08-22T14:30:00.031779072Z,177,30,NaN,C,A,6.67,70.0,NaN,0,6.62,6.67,6,70,0,0,SPY 230828C00435000
14436,1692714600031791104,1692714600031791104,2023-08-22T14:30:00.031791104Z,177,30,NaN,C,A,146.31,44.0,NaN,0,145.12,146.31,44,44,0,0,SPY 230929C00295000
14437,1692714600031791104,1692714600031791104,2023-08-22T14:30:00.031791104Z,177,30,NaN,A,A,112.52,1.0,NaN,0,111.33,112.52,2,1,0,0,SPY 230929C00329000
14438,1692714600031791104,1692714600031791104,2023-08-22T14:30:00.031791104Z,177,30,NaN,C,A,2.39,96.0,NaN,0,2.36,2.39,144,96,0,0,SPY 230908C00447500
14439,1692714600031791872,1692714600031791872,2023-08-22T14:30:00.031791872Z,177,30,NaN,C,A,28.04,55.0,NaN,0,26.85,28.04,33,55,0,0,SPY 230831P00467000


In [3]:
# Compare Rust decoded trades parquet vs DataBento pull (10k window)

symbol = 'SPY   230830P00423000'
START = '2023-08-22T14:29:59.990000Z'
END   = '2023-08-22T14:30:00.500000Z'

trades_df = trades_df[trades_df['symbol'] == symbol]
db_trades = db_trades[db_trades['symbol'] == symbol]
print(trades_df.shape)
display(trades_df.head())
print(db_trades.shape)
display(db_trades.head())

(6, 19)


,ts_event,ts_recv,ts_event_utc,rtype,publisher_id,instrument_id,action,side,price,size,flags,ts_in_delta,bid_px_00,ask_px_00,bid_sz_00,ask_sz_00,bid_pb_00,ask_pb_00,symbol
0,1692714599999800576,1692714599999800576,2023-08-22T14:29:59.999800576Z,177,30,NaN,A,A,0.5,418.0,NaN,0,0.48,0.5,310,418,0,0,SPY 230830P00423000
1820,1692714600005955584,1692714600005955584,2023-08-22T14:30:00.005955584Z,177,30,NaN,C,A,0.5,191.0,NaN,0,0.48,0.5,5,191,0,0,SPY 230830P00423000
2045,1692714600006426368,1692714600006426368,2023-08-22T14:30:00.006426368Z,177,30,NaN,C,A,0.5,698.0,NaN,0,0.48,0.5,520,698,0,0,SPY 230830P00423000
2780,1692714600007849728,1692714600007849728,2023-08-22T14:30:00.007849728Z,177,30,NaN,C,A,0.5,1308.0,NaN,0,0.48,0.5,639,1308,0,0,SPY 230830P00423000
7427,1692714600014484480,1692714600014484480,2023-08-22T14:30:00.014484480Z,177,30,NaN,C,A,0.5,138.0,NaN,0,0.48,0.5,5,138,0,0,SPY 230830P00423000


(8, 17)


,ts_event,rtype,publisher_id,instrument_id,action,side,price,size,flags,ts_in_delta,bid_px_00,ask_px_00,bid_sz_00,ask_sz_00,bid_pb_00,ask_pb_00,symbol
ts_recv,,,,,,,,,,,,,,,,,
2023-08-22 14:29:59.991083538+00:00,2023-08-22 14:29:59.990872832+00:00,177,30,654312856,A,N,0.49,271,194,0,0.48,0.49,1497,271,0,0,SPY 230830P00423000
2023-08-22 14:30:00.048469173+00:00,2023-08-22 14:30:00.048260352+00:00,177,30,654312856,A,N,0.49,271,194,0,0.48,0.49,1670,271,0,0,SPY 230830P00423000
2023-08-22 14:30:00.075664939+00:00,2023-08-22 14:30:00.075455488+00:00,177,30,654312856,A,N,0.49,321,194,0,0.48,0.49,1670,321,0,0,SPY 230830P00423000
2023-08-22 14:30:00.106431944+00:00,2023-08-22 14:30:00.106223616+00:00,177,30,654312856,A,N,0.49,321,194,0,0.48,0.49,1423,321,0,0,SPY 230830P00423000
2023-08-22 14:30:00.449679234+00:00,2023-08-22 14:30:00.449470208+00:00,177,30,654312856,A,N,0.49,271,194,0,0.48,0.49,1423,271,0,0,SPY 230830P00423000
